# Portfolio Analysis Project Assignment


In [88]:
import yfinance as yf
import pandas as pd
import numpy as np

In [62]:
selected_seven = ["META", "GOOGL", "DIS", "BBY", "AAL","CAT", "WRB"]

In [63]:
stocks = yf.download(selected_seven, start="2016-03-24", end="2026-03-24")


[*********************100%***********************]  7 of 7 completed


In [64]:
stocks.head()

Price           Close                                                          \
Ticker            AAL        BBY        CAT        DIS      GOOGL        META   
Date                                                                            
2016-03-24  38.809246  22.241356  59.527214  89.611252  37.432621  112.169754   
2016-03-28  39.202129  22.305330  59.550903  90.413170  37.355263  112.804764   
2016-03-29  39.623760  22.532787  60.096470  90.477684  37.980591  115.235695   
2016-03-30  39.652512  22.440380  60.491783  91.168991  38.102089  113.806900   
2016-03-31  39.297955  23.058786  60.515503  91.537682  37.832317  113.211578   

Price                       High                        ...       Open  \
Ticker            WRB        AAL        BBY        CAT  ...      GOOGL   
Date                                                    ...              
2016-03-24  12.713226  40.198714  22.504356  59.574650  ...  37.252114   
2016-03-28  12.859602  39.614178  22.539899  59.732753  ...  37.498575   
2016-03-29  13.017780  39.700422  22.575436  60.128096  ...  37.375095   
2016-03-30  13.256225  40.313705  22.767354  61.076854  ...  38.095642   
2016-03-31  13.268028  39.939986  23.442626  60.981977  ...  38.102087   

Price                                Volume                             \
Ticker            META        WRB       AAL      BBY      CAT      DIS   
Date                                                                     
2016-03-24  111.256912  12.736834  12515100  3007300  7036100  5577700   
2016-03-28  112.437642  12.521999   6402800  2851500  3425400  5317100   
2016-03-29  112.943684  12.850159   6794000  4114500  4085000  5170800   
2016-03-30  115.821100  13.083881   6076800  3084800  4886800  5677500   
2016-03-31  113.806905  13.220811   8138000  6364900  4348700  7038800   

Price                                    
Ticker         GOOGL      META      WRB  
Date                                     
2016-03-24  31116000  17839100  1547775  
2016-03-28  21676000  21351800  1091138  
2016-03-29  40062000  29825300  1724288  
2016-03-30  40396000  33235200  2672663  
2016-03-31  32554000  21207500  2737800  

[5 rows x 35 columns]

In [65]:
price_field="Close"

stock_10y = None
if isinstance(stocks.columns, pd.MultiIndex):
    # yfinance returns multiindex columns if multiple tickers
    if price_field not in stocks.columns.levels[0]:
        raise ValueError(f"Price field '{price_field}' not found. Available: {stocks.columns.levels[0]}")
    stock_10y = stocks[price_field].copy()
else:
    # single ticker case
    stock_10y = stocks[[price_field]].copy()
    stock_10y.columns = selected_seven

stock_10y = stock_10y.dropna(how="all").dropna(axis=1)  # drop any tickers that failed
if stock_10y.shape[1] != len(selected_seven):
    missing = set(selected_seven) - set(stock_10y.columns)
    raise ValueError(f"Missing tickers data: {missing}")

In [66]:
stock_10y.head()

Ticker,AAL,BBY,CAT,DIS,GOOGL,META,WRB
Date,,,,,,,
2016-03-24,38.809246,22.241356,59.527214,89.611252,37.432621,112.169754,12.713226
2016-03-28,39.202129,22.305330,59.550903,90.413170,37.355263,112.804764,12.859602
2016-03-29,39.623760,22.532787,60.096470,90.477684,37.980591,115.235695,13.017780
2016-03-30,39.652512,22.440380,60.491783,91.168991,38.102089,113.806900,13.256225
2016-03-31,39.297955,23.058786,60.515503,91.537682,37.832317,113.211578,13.268028


In [67]:
print(stock_10y.columns)

Index(['AAL', 'BBY', 'CAT', 'DIS', 'GOOGL', 'META', 'WRB'], dtype='object', name='Ticker')


In [77]:


stock_vs_market = pd.DataFrame({
    "portfolio_weight": [1/len(selected_seven)] * len(selected_seven),
    "annualized_volatility": stock_10y.tail(63).pct_change().std() * (252**0.5)

})



In [78]:
stock_vs_market

,portfolio_weight,annualized_volatility
Ticker,,
AAL,0.142857,0.480299
BBY,0.142857,0.354384
CAT,0.142857,0.355100
DIS,0.142857,0.282701
GOOGL,0.142857,0.208616
META,0.142857,0.348677
WRB,0.142857,0.205110


In [79]:
# SPY, IWM, DIA data pull
market_index = yf.download(["SPY", "IWM", "DIA"], start="2016-03-24", end="2026-03-24")["Close"].dropna()
market_index.head()

[*********************100%***********************]  3 of 3 completed


Ticker,DIA,IWM,SPY
Date,,,
2016-03-24,143.546982,94.282532,172.568619
2016-03-28,143.694702,94.396835,172.670578
2016-03-29,144.474228,97.017738,174.267776
2016-03-30,145.147095,97.026535,175.032425
2016-03-31,144.941864,97.299194,174.607590


In [80]:
# beta calculation: trailing 12 months (approx. 252 trading days)
# Align dates before computing cov/var.
trailing_days = 252
stock_12m = stock_10y.tail(trailing_days)
market_12m = market_index.tail(trailing_days).copy()
stock_ret_12m = stock_12m.pct_change()

for index in ["SPY", "IWM", "DIA"]:
    market_ret_12m = market_12m[index].pct_change()
    aligned = pd.concat([stock_ret_12m, market_ret_12m.rename('market')], axis=1).dropna()
    market_series = aligned['market']
    market_var = market_series.var()
    betas = aligned[selected_seven].apply(
        lambda s, m=market_series, mv=market_var: (s.cov(m) / mv) if mv != 0 else float('nan')
    )
    stock_vs_market[f"beta_{index}"] = betas.reindex(selected_seven).values

In [81]:
stock_vs_market

,portfolio_weight,annualized_volatility,beta_SPY,beta_IWM,beta_DIA
Ticker,,,,,
AAL,0.142857,0.480299,1.404413,0.929317,1.341197
BBY,0.142857,0.354384,0.978584,0.631370,0.892492
CAT,0.142857,0.355100,1.068486,0.831052,1.222611
DIS,0.142857,0.282701,1.380331,1.226335,1.601590
GOOGL,0.142857,0.208616,1.872054,1.707431,2.161746
META,0.142857,0.348677,1.215458,1.076918,1.385763
WRB,0.142857,0.205110,0.199997,0.150737,0.375212


In [82]:
# average weekly drawdown calculation
weekly_returns = stock_10y.resample('W').ffill().pct_change()
weekly_drawdown = (weekly_returns + 1).cumprod() - 1
average_weekly_drawdown = weekly_drawdown.mean()
stock_vs_market["average_weekly_drawdown"] = average_weekly_drawdown.values
stock_vs_market

,portfolio_weight,annualized_volatility,beta_SPY,beta_IWM,beta_DIA,average_weekly_drawdown
Ticker,,,,,,
AAL,0.142857,0.480299,1.404413,0.929317,1.341197,-0.380145
BBY,0.142857,0.354384,0.978584,0.631370,0.892492,1.864015
CAT,0.142857,0.355100,1.068486,0.831052,1.222611,2.457436
DIS,0.142857,0.282701,1.380331,1.226335,1.601590,0.262663
GOOGL,0.142857,0.208616,1.872054,1.707431,2.161746,1.852506
META,0.142857,0.348677,1.215458,1.076918,1.385763,1.596887
WRB,0.142857,0.205110,0.199997,0.150737,0.375212,1.703162


In [83]:
# maximum weekly drawdown calculation
weekly_max_drawdown = weekly_drawdown.min()
stock_vs_market["max_weekly_drawdown"] = weekly_max_drawdown.values
stock_vs_market

,portfolio_weight,annualized_volatility,beta_SPY,beta_IWM,beta_DIA,average_weekly_drawdown,max_weekly_drawdown
Ticker,,,,,,,
AAL,0.142857,0.480299,1.404413,0.929317,1.341197,-0.380145,-0.767066
BBY,0.142857,0.354384,0.978584,0.631370,0.892492,1.864015,-0.050290
CAT,0.142857,0.355100,1.068486,0.831052,1.222611,2.457436,-0.062940
DIS,0.142857,0.282701,1.380331,1.226335,1.601590,0.262663,-0.135100
GOOGL,0.142857,0.208616,1.872054,1.707431,2.161746,1.852506,-0.092258
META,0.142857,0.348677,1.215458,1.076918,1.385763,1.596887,-0.196904
WRB,0.142857,0.205110,0.199997,0.150737,0.375212,1.703162,0.020799


In [84]:
# total return - trailing 10-years
total_return = (stock_10y.iloc[-1] / stock_10y.iloc[0]) - 1
stock_vs_market["total_return"] = total_return.values
stock_vs_market

,portfolio_weight,annualized_volatility,beta_SPY,beta_IWM,beta_DIA,average_weekly_drawdown,max_weekly_drawdown,total_return
Ticker,,,,,,,,
AAL,0.142857,0.480299,1.404413,0.929317,1.341197,-0.380145,-0.767066,-0.721458
BBY,0.142857,0.354384,0.978584,0.631370,0.892492,1.864015,-0.050290,1.877972
CAT,0.142857,0.355100,1.068486,0.831052,1.222611,2.457436,-0.062940,10.787886
DIS,0.142857,0.282701,1.380331,1.226335,1.601590,0.262663,-0.135100,0.093055
GOOGL,0.142857,0.208616,1.872054,1.707431,2.161746,1.852506,-0.092258,7.069432
META,0.142857,0.348677,1.215458,1.076918,1.385763,1.596887,-0.196904,4.385231
WRB,0.142857,0.205110,0.199997,0.150737,0.375212,1.703162,0.020799,4.156048


In [85]:
# annualized total return - trailing 10-years
annualized_return = (stock_10y.iloc[-1] / stock_10y.iloc[0]) ** (252 / len(stock_10y)) - 1
stock_vs_market["annualized_return"] = annualized_return.values
stock_vs_market

,portfolio_weight,annualized_volatility,beta_SPY,beta_IWM,beta_DIA,average_weekly_drawdown,max_weekly_drawdown,total_return,annualized_return
Ticker,,,,,,,,,
AAL,0.142857,0.480299,1.404413,0.929317,1.341197,-0.380145,-0.767066,-0.721458,-0.120300
BBY,0.142857,0.354384,0.978584,0.631370,0.892492,1.864015,-0.050290,1.877972,0.111825
CAT,0.142857,0.355100,1.068486,0.831052,1.222611,2.457436,-0.062940,10.787886,0.280684
DIS,0.142857,0.282701,1.380331,1.226335,1.601590,0.262663,-0.135100,0.093055,0.008962
GOOGL,0.142857,0.208616,1.872054,1.707431,2.161746,1.852506,-0.092258,7.069432,0.232926
META,0.142857,0.348677,1.215458,1.076918,1.385763,1.596887,-0.196904,4.385231,0.183925
WRB,0.142857,0.205110,0.199997,0.150737,0.375212,1.703162,0.020799,4.156048,0.178773


In [89]:
# comparing the portfolio against the 3 market indices

weights = np.array([1/len(stock_10y.columns)] * len(stock_10y.columns))

stock_returns = stock_10y.pct_change().dropna()

portfolio_returns = stock_returns.dot(weights)

# current risk free rate
risk_free_rate = 0.03

portfolio_vs_market = pd.DataFrame({
    # a. ETF Ticker for column 1
    "ticker" : ["SPY", "IWM", "DIA"],
    # b. Correlation against ETF for column 2
    "correlation": [
            portfolio_returns.corr(market_index[col].pct_change())
            for col in market_index.columns
        ],    
    # c. covariance of portfolio against ETF for column 3
    "covariance": [
            portfolio_returns.cov(market_index[col].pct_change())
            for col in market_index.columns
        ],    
    # d. Tracking errors - trailing 10-years
    "tracking_error": [
            (portfolio_returns - market_index[col].pct_change()).std() * np.sqrt(252)
            for col in market_index.columns
        ],
    # e. Sharpe Ratio - using current risk-free rate
    "sharpe_ratio": [
            ((portfolio_returns.mean() - risk_free_rate/252) / portfolio_returns.std()) * np.sqrt(252)
        ] * 3,
    # f. Annualized Volatility (252 days) Spread (Portfolio Volatility – ETF Volatility)
    "annualized_volatility_spread": [
        portfolio_returns.std()*np.sqrt(252) - market_index[col].pct_change().std()*np.sqrt(252)
        for col in market_index.columns
    ]
})

portfolio_vs_market

,ticker,correlation,covariance,tracking_error,sharpe_ratio,annualized_volatility_spread
0,SPY,0.858873,0.000138,0.120630,0.623285,0.056160
1,IWM,0.836999,0.000176,0.131512,0.623285,0.001321
2,DIA,0.877304,0.000144,0.113376,0.623285,0.052107
